# Preparing Data

In [15]:
%pip install gradio insightface onnxruntime opencv-python-headless pillow numpy "numpy<2" pandas tqdm open_clip_torch torch --quiet

Note: you may need to restart the kernel to use updated packages.


##### Load two models: CLIP for appearance similarity and ArcFace for facial structure

In [2]:
import os
import json
import pickle
import datetime
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import open_clip
import gradio as gr
from PIL import Image
from tqdm.auto import tqdm
from insightface.app import FaceAnalysis

# ArcFace
print("Loading ArcFace (buffalo_s)...")
app = FaceAnalysis(name="buffalo_s", providers=["CPUExecutionProvider"])
app.prepare(ctx_id=0, det_size=(320, 320))

# CLIP
print("Loading CLIP (ViT-B/32)...")
clip_device = "cpu"
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
clip_model.eval().to(clip_device)

print("Both models ready.")

/Users/brandley/.pyenv/versions/3.11.3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading ArcFace (buffalo_s)...
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/brandley/.insightface/models/buffalo_s/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/brandley/.insightface/models/buffalo_s/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/brandley/.insightface/models/buffalo_s/det_500m.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/brandley/.insightface/models/buffalo_s/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /Users/brandley/.insightface/models/buffalo_s/w600k_mbf.onnx r

In [16]:
# File paths — adjust if you saved things to different locations
IMAGES_DIR        = Path("images")
ARC_FILE          = Path("app_data/player_embeddings.npz")
CLIP_FILE         = Path("app_data/clip_player_embeddings.npz")
CHECKPOINT_FILE   = Path("per_image_checkpoint.pkl")

# Outlier rejection parameters (same as your original Cell 5)
OUTLIER_THRESH    = 0.30
MIN_KEEP          = 3
MIN_FACE_SIZE     = 80
MAX_IMAGE_SIDE    = 1024

# Combined-model weights
TOP_K       = 3
ARC_WEIGHT  = 0.3
CLIP_WEIGHT = 0.7

In [17]:
# ---- Load ArcFace per-player means ----
arc_data = np.load(ARC_FILE)
clean = {k: arc_data[k] for k in arc_data.files}
print(f"Loaded {len(clean)} ArcFace fingerprints from {ARC_FILE}")

# ---- Load CLIP per-player means ----
clip_data = np.load(CLIP_FILE)
clip_clean = {k: clip_data[k] for k in clip_data.files}
print(f"Loaded {len(clip_clean)} CLIP fingerprints from {CLIP_FILE}")

# ---- Load per_image dict (ArcFace per-image vectors) from the checkpoint ----
if CHECKPOINT_FILE.exists():
    with open(CHECKPOINT_FILE, "rb") as f:
        per_image, no_face, done_paths = pickle.load(f)
    print(f"Loaded per_image from checkpoint: "
          f"{sum(len(v) for v in per_image.values())} vectors across "
          f"{len(per_image)} players")
else:
    raise FileNotFoundError(
        f"Could not find {CHECKPOINT_FILE}. You need this for best_kept_image() "
        f"to know which photos passed the outlier filter."
    )

# ---- Recompute kept_map from per_image ----
# (kept_map wasn't saved to disk separately; rebuild it here using the same
#  logic that originally created it, so best_kept_image() works correctly.)
kept_map = {}
for player, items in per_image.items():
    if not items:
        continue
    files = [f for f, _ in items]
    embs  = np.stack([e for _, e in items])

    if len(embs) == 1:
        kept_map[player] = files
        continue

    mean = embs.mean(axis=0)
    mean /= np.linalg.norm(mean) + 1e-9
    sims = embs @ mean

    keep = sims >= OUTLIER_THRESH
    if keep.sum() < MIN_KEEP:
        idx = np.argsort(-sims)[:MIN_KEEP]
        keep = np.zeros(len(embs), dtype=bool)
        keep[idx] = True

    kept_map[player] = [f for f, k in zip(files, keep) if k]

print(f"Rebuilt kept_map for {len(kept_map)} players.")

# Sanity check
common = set(clean.keys()) & set(clip_clean.keys()) & set(per_image.keys())
print(f"\nPlayers present in all three data structures: {len(common)}")

Loaded 1205 ArcFace fingerprints from app_data/player_embeddings.npz
Loaded 1269 CLIP fingerprints from app_data/clip_player_embeddings.npz
Loaded per_image from checkpoint: 17151 vectors across 1205 players
Rebuilt kept_map for 1205 players.

Players present in all three data structures: 1205


In [13]:
def best_kept_image(player_key):
    """Return the filename of the highest-scoring kept image for a player."""
    items = per_image[player_key]
    kept_set = set(kept_map[player_key])
    mean = clean[player_key]
    candidates = [(f, float(e @ mean)) for f, e in items if f in kept_set]
    if not candidates:
        return None
    return max(candidates, key=lambda x: x[1])[0]


@torch.no_grad()
def clip_embed_path(img_path):
    """Return the L2-normalized 512-d CLIP image embedding."""
    with Image.open(img_path) as pil:
        pil = pil.convert("RGB")
        tensor = clip_preprocess(pil).unsqueeze(0).to(clip_device)
    feat = clip_model.encode_image(tensor)
    feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.squeeze(0).cpu().numpy()


# Pre-stack the per-player matrices once so per-request matching is fast
COMMON_KEYS = sorted(set(clean.keys()) & set(clip_clean.keys()))
ARC_MATRIX  = np.stack([clean[k]      for k in COMMON_KEYS])
CLIP_MATRIX = np.stack([clip_clean[k] for k in COMMON_KEYS])
print(f"Pre-stacked matrices for {len(COMMON_KEYS)} players.")


def find_lookalike_combined_path(selfie_path, top_k=TOP_K):
    """Return top-k matches as (player_key, combined_score, arc_score, clip_score)."""
    # ArcFace embedding
    img = np.array(Image.open(selfie_path).convert("RGB"))
    faces = app.get(img)
    if not faces:
        return None
    faces.sort(key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1]),
               reverse=True)
    q_arc = faces[0].normed_embedding

    # CLIP embedding
    q_clip = clip_embed_path(selfie_path)

    # Score against every player
    arc_scores  = ARC_MATRIX  @ q_arc
    clip_scores = CLIP_MATRIX @ q_clip

    # Normalize to [0, 1] so the weights mean what we expect
    arc_n  = (arc_scores  - arc_scores.min())  / (arc_scores.ptp()  + 1e-9)
    clip_n = (clip_scores - clip_scores.min()) / (clip_scores.ptp() + 1e-9)
    combined = ARC_WEIGHT * arc_n + CLIP_WEIGHT * clip_n

    top = np.argsort(-combined)[:top_k]
    return [(COMMON_KEYS[i],
             float(combined[i]),
             float(arc_scores[i]),
             float(clip_scores[i])) for i in top]


def gradio_lookalike_combined(selfie_path):
    """Gradio handler — returns three (image, caption) pairs + a status message."""
    blank = [None, "", None, "", None, ""]
    if selfie_path is None:
        return [*blank, "Please upload a selfie first."]

    results = find_lookalike_combined_path(selfie_path)
    if results is None:
        return [*blank, "No face detected — try a clearer, forward-facing photo."]

    images   = [None, None, None]
    captions = ["",   "",   ""]
    for i, (player_key, combined, arc_s, clip_s) in enumerate(results):
        team, name = player_key.split("__", 1)
        rep_file = best_kept_image(player_key)
        if not rep_file:
            continue
        rep_path = IMAGES_DIR / team / name / rep_file
        if not rep_path.exists():
            continue
        pct = int(round(combined * 100))
        images[i] = str(rep_path)
        captions[i] = (
            f"### {name.replace('_', ' ')}\n"
            f"**{team}**  \n"
            f"{pct}% match"
        )
    return [images[0], captions[0],
            images[1], captions[1],
            images[2], captions[2],
            ""]

Pre-stacked matrices for 1205 players.


In [14]:
arc_pct  = int(round(ARC_WEIGHT  * 100))
clip_pct = int(round(CLIP_WEIGHT * 100))

# Cleanly close any previous Gradio server before launching a new one
try:
    demo.close()
except Exception:
    pass

with gr.Blocks(
    title=f"Footballer Lookalike — ArcFace {arc_pct}% / CLIP {clip_pct}%",
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown(
        f"# Which footballer do you look like? ⚽️👤  "
        f"<span style='font-size:0.55em; opacity:0.65; font-weight:normal;'>"
        f"ArcFace {arc_pct}% &nbsp;·&nbsp; CLIP {clip_pct}%</span>"
    )
    gr.Markdown(
        "Upload a clear, forward-facing selfie or use your camera. "
        "We combine **facial structure** (ArcFace) with **appearance similarity** "
        "(CLIP) to find your closest match. Your picture is not saved or shared."
    )

    with gr.Row():
        with gr.Column(scale=1):
            selfie_in  = gr.Image(type="filepath", label="Your selfie",
                                  sources=["upload", "webcam"])
            submit_btn = gr.Button("Find my lookalikes", variant="primary")

        with gr.Column(scale=3):
            with gr.Row():
                with gr.Column():
                    img1 = gr.Image(show_label=False, height=240, interactive=False)
                    cap1 = gr.Markdown("")
                with gr.Column():
                    img2 = gr.Image(show_label=False, height=240, interactive=False)
                    cap2 = gr.Markdown("")
                with gr.Column():
                    img3 = gr.Image(show_label=False, height=240, interactive=False)
                    cap3 = gr.Markdown("")
            status_out = gr.Markdown("")

    gr.Markdown(
        f"<div style='text-align:center; font-size:0.85em; opacity:0.6; "
        f"margin-top:1em;'>Model weights: ArcFace {arc_pct}% &nbsp;·&nbsp; "
        f"CLIP {clip_pct}%</div>"
    )

    submit_btn.click(
        gradio_lookalike_combined,
        inputs=[selfie_in],
        outputs=[img1, cap1, img2, cap2, img3, cap3, status_out]
    )

demo.launch()

/var/folders/0g/v46xc7wn14g9tc8__fwrprm00000gn/T/ipykernel_23953/3020869929.py:10: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Closing server running on port: 7861
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
